# Data Project: Inequality in Denmark

**Group:** Kristian, Kasper, David og Bjørn

This project looks at how income inequality in Denmark has developed over time, using two tables from Statistics Denmark (IFOR41 and IFOR32). Both are based on equivalised disposable income (income after tax and transfers, adjusted for household size). The project has two parts: an empirical analysis of real data (section 1) and a simulation of the income distribution (section 2).

## Setup

We install and import the packages we need, plus our own module `opgave1.py`, which holds the functions for downloading and cleaning data from Statistics Denmark.

In [9]:
# install dstapi (run once, then comment out)
#%pip install git+https://github.com/alemartinello/dstapi

In [10]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize
import opgave1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'dstapi'

# 1. Inequality in Denmark

## 1.1 The Gini coefficient and the top-10% share

We use two measures of income inequality for Denmark over time:

- The **Gini coefficient** from table IFOR41, where the inequality measure is picked with `ULLIG = 70`. Gini runs from 0 (everyone equal) to 100 (one person has all the income).
- The **top-10% share**, calculated from the decile groups in table IFOR32.

All downloading and cleaning is done through the `load_dst` function in `opgave1.py`, so getting a new table only takes one function call.

### Gini coefficient (IFOR41)

We get the Gini for all of Denmark (`KOMMUNEDK = 000`) for every year.

In [ ]:
gini_dk = opgave1.load_dst(
    'IFOR41',
    variables=[
        {'code': 'ULLIG',     'values': ['70']},   # Gini coefficient
        {'code': 'KOMMUNEDK', 'values': ['000']},  # all of Denmark
        {'code': 'Tid',       'values': ['*']},     # all years
    ],
    value_name='gini',
)
gini_dk.head()

### Top-10% share (IFOR32)

IFOR32 gives the average income in each of the ten decile groups. Since each decile holds the same number of people, the top decile's share of total income is:

$$\text{top 10 pct.} = \frac{\bar{y}_{10}}{\sum_{d=1}^{10}\bar{y}_d}$$

We first get all ten deciles for Denmark.

In [ ]:
deciles_dk = opgave1.load_dst(
    'IFOR32',
    variables=[
        {'code': 'DECILGEN',  'values': ['*']},    # all 10 deciles
        {'code': 'KOMMUNEDK', 'values': ['000']},  # all of Denmark
        {'code': 'Tid',       'values': ['*']},     # all years
    ],
    value_name='avg_income',
)
deciles_dk.head(12)

Then we calculate the top-10% share by dividing the tenth decile's income by the sum of all ten deciles, for each year.

In [ ]:
# denominator: total income per year (sum over all 10 deciles)
total_per_year = deciles_dk.groupby('year')['avg_income'].sum()

# numerator: the tenth decile's income per year
top_decile = deciles_dk[deciles_dk['DECILGEN'] == 'Tenth decil']['avg_income']

# top-10% share = tenth decile / sum of all deciles
top10 = pd.DataFrame({'top10': top_decile / total_per_year})
top10.head()

### Combining the two measures

We merge the two series on year. `validate='1:1'` checks that there is exactly one row per year in each series - if not, pandas raises an error instead of silently duplicating rows.

In [ ]:
df_dk = pd.merge(
    gini_dk,          # column 'gini'
    top10,            # column 'top10'
    left_index=True,  # match on the year index in both
    right_index=True,
    validate='1:1',   # check: exactly one row per year in each
)
df_dk.head()

### Figure: development over time

The two measures are on very different scales (Gini around 22-30, the share around 0.18-0.25), so we plot them on separate y-axes.

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

# left axis: Gini
color1 = 'tab:blue'
ax1.plot(df_dk.index, df_dk['gini'], color=color1, label='Gini coefficient')
ax1.set_xlabel('Year')
ax1.set_ylabel('Gini coefficient', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

# right axis: top-10% share
ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.plot(df_dk.index, df_dk['top10'], color=color2, label='Top-10% share')
ax2.set_ylabel('Top-10% share', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

fig.suptitle('Income inequality in Denmark, 1987-2024')
fig.tight_layout()
plt.show()

### Correlation and interpretation

We calculate the Pearson correlation between the two measures to see if they tell the same story.

In [ ]:
corr = df_dk['gini'].corr(df_dk['top10'])
print(f'Correlation between Gini and top-10% share: {corr:.3f}')

**Interpretation:** Both measures show a clear and steady rise in income inequality in Denmark since 1987. The Gini coefficient has gone from about 22 to about 30, and the top-10% share from about 0.185 to about 0.25 - so the richest 10% take a growing slice of total income. The two series track each other closely across the whole period, which shows up as a very high correlation (close to 1). The fact that two separately built measures move almost identically makes us more confident that this is a real change in inequality, and not just a quirk of one particular measure.

## 1.2 Prediction

We fit a linear and a quadratic trend to the Gini coefficient and use them to predict inequality in 2030 and 2040. We solve the least-squares problem **ourselves** with `scipy.optimize.minimize`, and check it against `np.polyfit`.

### Preparing the data

We center the years around their mean. Without centering, $t^2$ would be very large numbers (e.g. $1987^2 \approx 3.9$ million), which makes the optimization numerically messy. Centering doesn't change the shape of the curve, only how we read the constant term.

In [ ]:
years = df_dk.index.values.astype(float)
gini_vals = df_dk['gini'].values

# center the years so the numbers are small and numerically stable
t = years - years.mean()

print("Number of points:", len(t))
print("t ranges from", t.min(), "to", t.max())

### Linear trend

We write the objective function ourselves (the sum of squared residuals) and minimize it. As a check, we solve the same problem analytically with `np.polyfit`; the two should give almost identical coefficients.

In [ ]:
# objective function: sum of squared residuals (SSR)
def ssr_linear(beta, t, y):
    b0, b1 = beta
    prediction = b0 + b1 * t
    residuals = y - prediction
    return np.sum(residuals**2)

# minimize SSR with scipy
result_lin = optimize.minimize(
    ssr_linear,
    x0=[0, 0],           # starting guess for [b0, b1]
    args=(t, gini_vals), # extra arguments passed to ssr_linear
)
b0_lin, b1_lin = result_lin.x
print(f"Linear trend (minimize):  b0 = {b0_lin:.4f},  b1 = {b1_lin:.4f}")

# check with np.polyfit (returns [b1, b0])
poly_lin = np.polyfit(t, gini_vals, deg=1)
print(f"Linear trend (polyfit):   b0 = {poly_lin[1]:.4f},  b1 = {poly_lin[0]:.4f}")

### Quadratic trend

Same approach, but with an extra term $\beta_2 t^2$ that can capture whether the rise speeds up or flattens out.

In [ ]:
def ssr_quad(beta, t, y):
    b0, b1, b2 = beta
    prediction = b0 + b1 * t + b2 * t**2
    residuals = y - prediction
    return np.sum(residuals**2)

result_quad = optimize.minimize(
    ssr_quad,
    x0=[0, 0, 0],
    args=(t, gini_vals),
)
b0_q, b1_q, b2_q = result_quad.x
print(f"Quadratic trend (minimize):  b0 = {b0_q:.4f},  b1 = {b1_q:.4f},  b2 = {b2_q:.4f}")

# check with np.polyfit (returns [b2, b1, b0])
poly_q = np.polyfit(t, gini_vals, deg=2)
print(f"Quadratic trend (polyfit):   b0 = {poly_q[2]:.4f},  b1 = {poly_q[1]:.4f},  b2 = {poly_q[0]:.4f}")

### Predictions for 2030 and 2040

Important: since the models are fitted on **centered** years, future years have to be centered with the **same** mean before we plug them into the model.

In [ ]:
year_mean = years.mean()
future_years = np.array([2030, 2040])
t_future = future_years - year_mean

pred_lin = b0_lin + b1_lin * t_future
pred_quad = b0_q + b1_q * t_future + b2_q * t_future**2

for yr, pl, pq in zip(future_years, pred_lin, pred_quad):
    print(f"{yr}:  linear = {pl:.2f},  quadratic = {pq:.2f}")

### Figure: data, trends and predictions

The dashed line marks where the actual data ends (2024). Everything to the right is extrapolation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# actual data
ax.scatter(years, gini_vals, color='black', s=25, label='Actual data', zorder=3)

# smooth grid for drawing the trends
year_grid = np.linspace(1987, 2040, 200)
t_grid = year_grid - year_mean

ax.plot(year_grid, b0_lin + b1_lin * t_grid,
        color='tab:blue', label='Linear trend')
ax.plot(year_grid, b0_q + b1_q * t_grid + b2_q * t_grid**2,
        color='tab:red', label='Quadratic trend')

# mark the predictions
ax.scatter(future_years, pred_lin, color='tab:blue', marker='D', s=60, zorder=4)
ax.scatter(future_years, pred_quad, color='tab:red', marker='D', s=60, zorder=4)

# vertical line where data ends
ax.axvline(years.max(), color='gray', linestyle='--', alpha=0.6)

ax.set_xlabel('Year')
ax.set_ylabel('Gini coefficient')
ax.set_title('Gini coefficient with linear and quadratic trend and predictions')
ax.legend()
fig.tight_layout()
plt.show()

**Interpretation:** The Gini coefficient has risen steadily from about 22 in 1987 to about 30 in 2024, a clear increase in income inequality. The linear trend fits the data well, with a slope of about 0.26 Gini points per year. The quadratic trend only adds a tiny curvature term ($b_2 \approx 0.001$), so inside the data range the two models are almost the same. The difference only shows up in the predictions: the linear model predicts a Gini of 32.4 in 2030 and 35.0 in 2040, while the quadratic predicts 32.9 and 36.3. The quadratic is higher because the small positive curvature makes inequality speed up a bit. The gap between the models grows the further out we extrapolate, which is a reminder that predictions far beyond the data depend heavily on the model choice and should be read with caution.

## 1.3 Municipalities

We repeat the analysis at the municipality level by getting all municipalities (`KOMMUNEDK = *`), and look at the most/least unequal ones as well as the biggest/smallest changes in Gini over time.

**Note:** Gini for small municipalities can jump around a lot from year to year, so we always plot the underlying series before reading anything into the rankings.

### Get Gini for all municipalities

The only change from 1.1 is `KOMMUNEDK` from `['000']` to `['*']`. Since the geography variable now varies, `load_dst` automatically keeps the `KOMMUNEDK` column.

In [ ]:
gini_kom = opgave1.load_dst(
    'IFOR41',
    variables=[
        {'code': 'ULLIG',     'values': ['70']},   # Gini
        {'code': 'KOMMUNEDK', 'values': ['*']},     # ALL geographies
        {'code': 'Tid',       'values': ['*']},     # all years
    ],
    value_name='gini',
)
print("Shape:", gini_kom.shape)
gini_kom.head()

### Reshape to wide format

We use `pivot` to get years as rows and municipalities as columns. That makes per-municipality statistics (like the mean) a simple column operation.

In [ ]:
gini_wide = gini_kom.pivot(columns='KOMMUNEDK', values='gini')
print("Shape:", gini_wide.shape)   # (number of years, number of geographies)
gini_wide.iloc[:5, :5]

### Most and least unequal municipalities

We rank municipalities by their average Gini over the whole period (more robust than a single year). "All Denmark" is dropped since it isn't a municipality.

In [ ]:
# drop "All Denmark" — it isn't a municipality
kommuner = gini_wide.drop(columns='All Denmark')

# average Gini per municipality over the whole period
mean_gini = kommuner.mean().sort_values(ascending=False)

print("=== 10 MOST unequal municipalities (avg Gini) ===")
print(mean_gini.head(10))
print("\n=== 10 LEAST unequal municipalities (avg Gini) ===")
print(mean_gini.tail(10))

### Biggest change in inequality over time

We measure the change as Gini in the last year minus Gini in the first year, for each municipality.

In [ ]:
first_year = kommuner.index.min()   # 1987
last_year = kommuner.index.max()    # 2024

change = (kommuner.loc[last_year] - kommuner.loc[first_year]).sort_values(ascending=False)

print(f"Change in Gini from {first_year} to {last_year}\n")
print("=== 10 BIGGEST increases ===")
print(change.head(10))
print("\n=== 10 BIGGEST decreases / smallest increases ===")
print(change.tail(10))

### Checking the underlying series

Before reading anything into the ranking of changes, we plot the series for the municipalities with the biggest "increase" together with the national average. This shows whether a large measured change is a real, lasting trend or just a data break / noise in the endpoints.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for kom in ['Vejen', 'Rudersdal', 'Gentofte']:
    ax.plot(kommuner.index, kommuner[kom], label=kom)

# national average as reference
ax.plot(gini_wide.index, gini_wide['All Denmark'],
        color='black', linestyle='--', label='All Denmark')

ax.set_xlabel('Year')
ax.set_ylabel('Gini coefficient')
ax.set_title('Gini over time: municipalities with the biggest "increase"')
ax.legend()
fig.tight_layout()
plt.show()

**Interpretation:** At the municipality level, inequality varies much more than at the national level. The most unequal municipalities are consistently the wealthy ones north of Copenhagen - Gentofte (avg Gini ~40), Rudersdal and Hørsholm - where high top incomes create a wide spread. The least unequal are more homogeneous suburban and provincial municipalities like Egedal and Halsnæs (~21).

When we rank by the *change* in Gini from 1987 to 2024, Vejen tops the list with +25 points. But the figure shows this comes from a sudden jump in 2022-2023 rather than a lasting trend - most likely a data break. The other big increases (Rudersdal, Gentofte, Copenhagen) are gradual and believable, and stay well above the national average.

This is the point the assignment warns about: rankings based on two endpoints are sensitive to noise and data breaks in exactly those two years, so you should always look at the underlying series before drawing conclusions. A more robust measure of the trend would be the slope from a trend fitted to the whole period (as in section 1.2).

## 1.4 Extension: income level and inequality

As an extension we look at whether there is a link between a municipality's **income level** and its **inequality**: are richer municipalities also more unequal? We use IFOR32, which we already know, to work out each municipality's average income, and relate it to the average Gini from section 1.3.

### Get decile incomes for all municipalities

Same call as the IFOR32 one in 1.1, but with `KOMMUNEDK = *`.

In [ ]:
deciles_kom = opgave1.load_dst(
    'IFOR32',
    variables=[
        {'code': 'DECILGEN',  'values': ['*']},    # all 10 deciles
        {'code': 'KOMMUNEDK', 'values': ['*']},     # ALL municipalities
        {'code': 'Tid',       'values': ['*']},     # all years
    ],
    value_name='avg_income',
)
print("Shape:", deciles_kom.shape)
deciles_kom.head()

### Average income per municipality, joined with Gini

Since each decile group holds the same number of people, the mean of the ten deciles equals the municipality's average income. We also average over all years to get one robust number per municipality, and join it with `mean_gini` from 1.3.

In [ ]:
# average income per municipality (mean over deciles AND years)
income_per_kom = deciles_kom.groupby('KOMMUNEDK')['avg_income'].mean()

# put both measures in one table, one row per municipality
kom_summary = pd.DataFrame({
    'mean_income': income_per_kom,
    'mean_gini': mean_gini,        # from 1.3
})
kom_summary = kom_summary.drop(index='All Denmark', errors='ignore')

print("Number of municipalities:", len(kom_summary))
kom_summary.head()

### Scatter plot and correlation

Each municipality is one point. We label a few well-known ones so the plot is readable.

In [ ]:
corr_kom = kom_summary['mean_income'].corr(kom_summary['mean_gini'])
print(f"Correlation (income vs. Gini): {corr_kom:.3f}")

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(kom_summary['mean_income'], kom_summary['mean_gini'],
           alpha=0.6, edgecolor='k', linewidth=0.5)

for name in ['Gentofte', 'Rudersdal', 'Aarhus', 'Copenhagen', 'Egedal']:
    if name in kom_summary.index:
        x = kom_summary.loc[name, 'mean_income']
        y = kom_summary.loc[name, 'mean_gini']
        ax.annotate(name, (x, y), fontsize=9,
                    xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Average income (DKK)')
ax.set_ylabel('Average Gini coefficient')
ax.set_title('Income level vs. inequality across municipalities')
fig.tight_layout()
plt.show()

**Interpretation:** There is a clear positive link between a municipality's income level and its inequality (correlation ≈ 0.75): richer municipalities tend to be more unequal. The link is largely driven by the wealthy municipalities north of Copenhagen (Gentofte, Rudersdal, Hørsholm), which have both the highest incomes and the highest inequality. The reason is their income mix: people with very high capital and top incomes living alongside ordinary wage earners and pensioners. It's that spread - not the wealth itself - that pushes the Gini up.

The link isn't deterministic, though: Egedal has a high income level but low inequality, because incomes there are evenly spread. It's also worth noting that the correlation depends heavily on the few outliers in the top right; without them the link among the remaining municipalities would be much weaker. So the result should be read as a tendency, not a rule.

# 2. Simulating the income distribution

[Simulation part - builds on `opgave2.py`.]

# 2.1 Simulating an Income Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import opgave2
import importlib

importlib.reload(opgave2)

education, ages, state, human_capital, income = opgave2.simulate_model()


# 2.2 Simulate the Income Distribution

In [ ]:
## Checking the Simulation
education_shares = np.bincount(education) / opgave2.N

print("Education shares:")
print(education_shares)

unemployment_by_age = np.mean(state == 1, axis=0)

print("Unemployment by age:")
print(unemployment_by_age)

print("Theoretical unemployment rate:")
print(opgave2.sigma / (opgave2.sigma + opgave2.lambda_))


### Income over the Life Cycle

In [ ]:

mean_income = np.mean(income, axis=0)

p10 = np.percentile(income, 10, axis=0)
p25 = np.percentile(income, 25, axis=0)
p50 = np.percentile(income, 50, axis=0)
p75 = np.percentile(income, 75, axis=0)
p90 = np.percentile(income, 90, axis=0)

# Plot mean income and selected percentiles

plt.figure(figsize=(10, 6))

plt.plot(ages, mean_income, label="Mean", linewidth=2)
plt.plot(ages, p10, label="10th percentile")
plt.plot(ages, p25, label="25th percentile")
plt.plot(ages, p50, label="Median")
plt.plot(ages, p75, label="75th percentile")
plt.plot(ages, p90, label="90th percentile")

plt.xlabel("Age")
plt.ylabel("Income")
plt.title("Income over the Life Cycle")
plt.legend()
plt.grid(True)
plt.xlim(18, 65)

plt.tight_layout()
plt.show()

### Income Distribution at Different Ages

In [ ]:

for age in [25, 35, 45, 60]:

    age_index = np.where(ages == age)[0][0]

    income_at_age = income[:, age_index]

    plt.figure(figsize=(8, 5))

    plt.hist(income_at_age, bins=30)

    plt.xlabel("Income")
    plt.ylabel("Number of individuals")
    plt.title(f"Income Distribution at Age {age}")

    plt.show()

## 2.3 Compute the Gini Coefficient

We compute the Gini coefficient of the simulated income distribution.
We first define our own Gini function and test it on distributions
where the theoretical answer is known.

In [ ]:
def gini(x):

    # Sort incomes from lowest to highest
    x = np.sort(x)

    # Number of observations
    n = len(x)

    # Calculate the Gini coefficient
    gini_value = (2* np.sum((np.arange(1, n + 1)) * x)/ (n * np.sum(x))- (n + 1) / n)

    return gini_value

### Testing the Gini function

We test the function using a uniform distribution on [0, 1].
The theoretical Gini coefficient is 1/3.

In [ ]:
# Uniform distribution

uniform_income = np.linspace(0, 1, 10000)

gini_uniform = gini(uniform_income)

print("Gini for uniform distribution:")
print(gini_uniform)

print("Theoretical Gini:")
print(1 / 3)

### Lognormal distribution

We also test the Gini function on a lognormal distribution.

In [ ]:
# Lognormal distribution

s = 0.5

rng_test = np.random.default_rng(123)

lognormal_income = rng_test.lognormal(0, s, 10000)

gini_lognormal = gini(lognormal_income)

print("Gini for lognormal distribution:")
print(gini_lognormal)

### Gini coefficient for the full simulated sample

We calculate the Gini coefficient for all simulated income observations pooled together.

In [ ]:
# Gini coefficient for all individuals and ages pooled

all_income = []

for i in range(len(income)):
    for j in range(len(ages)):
        all_income.append(income[i, j])

# Calculate the pooled Gini coefficient

gini_pooled = gini(all_income)

print("Gini coefficient for all individuals and ages pooled:")
print(gini_pooled)


# Gini coefficient by age

gini_by_age = []

for i in range(len(ages)):
    
    income_at_age = income[:, i]
    
    gini_at_age = gini(income_at_age)
    
    gini_by_age.append(gini_at_age)

print("Gini coefficient by age:")

for i in range(len(ages)):
    print(f"Age {ages[i]}: {gini_by_age[i]:.3f}")


# Plot Gini by age

plt.figure(figsize=(10, 6))

plt.plot(ages, gini_by_age)

plt.xlabel("Age")
plt.ylabel("Gini coefficient")
plt.title("Income Inequality by Age")

plt.xlim(18, 65)
plt.grid(True)

plt.tight_layout()
plt.show()

### Lorenz Curve

The Lorenz curve shows the cumulative share of total income received by the cumulative share of the population.

In [ ]:
# Sort incomes from lowest to highest

sorted_income = sorted(all_income)

# Total income

total_income = sum(sorted_income)

# Cumulative income

cumulative_income = []

current_income = 0

for income_value in sorted_income:
    
    current_income += income_value
    
    cumulative_income.append(current_income / total_income)


# Cumulative population share

cumulative_population = []

for i in range(len(sorted_income)):
    
    cumulative_population.append((i + 1) / len(sorted_income))

plt.figure(figsize=(8, 6))

# Lorenz curve

plt.plot(cumulative_population,  cumulative_income, label="Lorenz curve")

# Line of perfect equality

plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect equality")

plt.xlabel("Cumulative share of population")
plt.ylabel("Cumulative share of income")
plt.title("Lorenz Curve")

plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Pooled versus within-age inequality

The pooled Gini coefficient measures inequality when individuals of all ages are considered together. It therefore captures both differences in income within age groups and differences in income across ages.

The Gini coefficients calculated separately by age measure inequality only among individuals of the same age. These coefficients show how income inequality develops over the life cycle.

The pooled Gini coefficient is higher than the inequality within most individual age groups. This indicates that differences across ages contribute to overall income inequality, in addition to differences between individuals of the same age.

In [ ]:
# Compare pooled inequality with inequality within age groups

print("Pooled Gini coefficient:")
print(f"{gini_pooled:.3f}")

print("\nGini coefficient for selected ages:")

for age in [25, 35, 45, 55, 65]:

    age_index = np.where(ages == age)[0][0]

    print(f"Age {age}: {gini_by_age[age_index]:.3f}")

## 2.4 What drives inequality?

We investigate which mechanisms in the model are important for generating income inequality.
We start by removing educational differences while keeping the other mechanisms unchanged.

In [ ]:
# Reload the updated model

import importlib
importlib.reload(opgave2)


# Run the model without educational differences

education_no, ages_no, state_no, human_capital_no, income_no = opgave2.simulate_no_education()

# Collect all income observations

all_income_no = []

for i in range(len(income_no)):
    for j in range(len(ages_no)):
        all_income_no.append(income_no[i, j])


# Calculate pooled Gini

gini_no_education = gini(all_income_no)

print("Pooled Gini without education differences:")
print(f"{gini_no_education:.3f}")

# Gini at age 45

age = 45

age_index = np.where(ages_no == age)[0][0]

gini_no_education_45 = gini(income_no[:, age_index])

print("Gini at age 45 without education differences:")
print(f"{gini_no_education_45:.3f}")

### Then we do the same for income shocks

In [ ]:
education_shock, ages_shock, state_shock, human_capital_shock, income_shock = opgave2.simulate_no_shocks()

# Collect all income observations

all_income_shock = []

for i in range(len(income_shock)):
    for j in range(len(ages_shock)):
        all_income_shock.append(income_shock[i, j])


# Calculate pooled Gini

gini_no_shocks = gini(all_income_shock)

print("Pooled Gini without human capital shocks:")
print(f"{gini_no_shocks:.3f}")

# Gini at age 45

age = 45

age_index = np.where(ages_shock == age)[0][0]

gini_no_shocks_45 = gini(income_shock[:, age_index])

print("Gini at age 45 without human capital shocks:")
print(f"{gini_no_shocks_45:.3f}")

### Then without depreciation of human capital

In [ ]:
importlib.reload(opgave2)
education_dep, ages_dep, state_dep, human_capital_dep, income_dep = opgave2.simulate_no_depreciation()
all_income_dep = []

for i in range(len(income_dep)):
    for j in range(len(ages_dep)):
        all_income_dep.append(income_dep[i, j])

gini_no_depreciation = gini(all_income_dep)

print("Pooled Gini without depreciation:")
print(f"{gini_no_depreciation:.3f}")

age = 45

age_index = np.where(ages_dep == age)[0][0]

gini_no_depreciation_45 = gini(income_dep[:, age_index])

print("Gini at age 45 without depreciation:")
print(f"{gini_no_depreciation_45:.3f}")

### Then we simulate where everyone is employed

In [ ]:
importlib.reload(opgave2)

education_unemp, ages_unemp, state_unemp, human_capital_unemp, income_unemp = opgave2.simulate_no_unemployment()

# Collect all income observations

all_income_unemp = []

for i in range(len(income_unemp)):
    for j in range(len(ages_unemp)):
        all_income_unemp.append(income_unemp[i, j])


# Calculate pooled Gini

gini_no_unemployment = gini(all_income_unemp)

print("Pooled Gini without unemployment:")
print(f"{gini_no_unemployment:.3f}")

# Gini at age 45

age = 45

age_index = np.where(ages_unemp == age)[0][0]

gini_no_unemployment_45 = gini(income_unemp[:, age_index])

print("Gini at age 45 without unemployment:")
print(f"{gini_no_unemployment_45:.3f}")

### Results

The baseline pooled Gini coefficient is 0.377. Removing educational differences reduces the pooled Gini to 0.298, showing that education is an important source of income inequality in the model.

Removing human capital shocks has the largest effect on the pooled Gini, which falls to 0.280. The Gini coefficient at age 45 also falls substantially to 0.215. This indicates that human capital shocks are an important source of both overall and within-age inequality.

Removing depreciation while unemployed increases the pooled Gini slightly from 0.377 to 0.382. Similarly, removing unemployment increases the pooled Gini to 0.387. Thus, in this model, depreciation and unemployment have a small equalising effect on the income distribution.

Overall, the results suggest that human capital shocks and educational differences are important drivers of inequality, while unemployment and depreciation reduce inequality slightly in the model.

## 2.5 Extension: Health Risk

We extend the model by introducing health risk as an additional source of uncertainty. The probability of becoming sick is low at young ages and increases with age. Sick individuals can recover in the following period, with the probability of recovery decreasing with age.

For employed individuals, being sick reduces income to 70% of their normal income.

In [ ]:
importlib.reload(opgave2)
education_health, ages_health, state_health, human_capital_health, income_health, healthy = opgave2.simulate_health_risk()
all_income_health = []

for i in range(len(income_health)):
    for j in range(len(ages_health)):
        all_income_health.append(income_health[i, j])

gini_health = gini(all_income_health)

print("Pooled Gini with health risk:")
print(f"{gini_health:.3f}")

age = 45

age_index = np.where(ages_health == age)[0][0]

gini_health_45 = gini(income_health[:, age_index])

print("Gini at age 45 with health risk:")
print(f"{gini_health_45:.3f}")

The pooled Gini coefficient is 0.376 with health risk, compared with 0.377 in the baseline model. Thus, the additional health risk has only a very small effect on overall income inequality in our simulation.

At age 45, the Gini coefficient is 0.339. This shows that health risk introduces some additional variation in income within the age group, although its overall effect on inequality is relatively limited.